# 📊 Lakeview Dashboard Analysis & Migration Preparation

This notebook provides step-by-step instructions for analyzing a Lakeview dashboard before migration.

## Purpose
Before migrating a dashboard, it's important to:
1. Identify all data dependencies (catalog.schema.table references)
2. Verify these objects exist in the target workspace
3. Plan catalog/schema mappings if needed
4. Validate query compatibility

## Prerequisites
- Access to both source and target Databricks workspaces
- Dashboard JSON file (exported from source workspace)
- Appropriate permissions to query Unity Catalog in target workspace

## 📦 Install Dependencies

In [ ]:
# Install required packages if not already installed
%pip install databricks-sdk --quiet

## 🔐 Authentication Setup

In [ ]:
# ============================================================================
# AUTHENTICATION CONFIGURATION
# ============================================================================

# Choose authentication method: "pat" or "oauth"
AUTH_METHOD = "pat"  # Change to "oauth" for Azure AD authentication

if AUTH_METHOD == "pat":
    # PAT tokens for source and target workspaces
    SOURCE_PAT_TOKEN = dbutils.secrets.get(scope="your-secret-scope", key="source_pat_token")
    TARGET_PAT_TOKEN = dbutils.secrets.get(scope="your-secret-scope", key="target_pat_token")
    
    # Alternative: Set directly (not recommended for production)
    # SOURCE_PAT_TOKEN = "dapi..."
    # TARGET_PAT_TOKEN = "dapi..."
else:
    # OAuth uses environment variables or Azure CLI credentials
    # Set these before running:
    # export ARM_CLIENT_ID="..."
    # export ARM_TENANT_ID="..."
    # export ARM_CLIENT_SECRET="..."
    print("Using OAuth authentication (Azure AD)")

## 📍 Step 1: Provide Original JSON Location

Specify the location of your dashboard JSON file. This can be:
- A file path in the workspace (e.g., `/Workspace/Users/user@example.com/dashboard.json`)
- A file path in Unity Catalog Volume
- A file path in DBFS
- Or you can export it directly from the source workspace in the next step

In [ ]:
# ============================================================================
# STEP 1: DASHBOARD JSON LOCATION
# ============================================================================

# Option 1: Local file path in workspace
DASHBOARD_JSON_PATH = "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/category_insights_dashboard.lvdash.json"

# Option 2: Unity Catalog Volume path
# DASHBOARD_JSON_PATH = "/Volumes/archana_krish_fe_dsa/vizient_deep_dive/data_files/dashboard.json"

# Option 3: DBFS path
# DASHBOARD_JSON_PATH = "/dbfs/mnt/dashboards/dashboard.json"

# Option 4: Export directly from source workspace (set these if using Option 4)
SOURCE_WORKSPACE_URL = "https://e2-demo-field-eng.cloud.databricks.com"
SOURCE_DASHBOARD_ID = "01f0fb1aabc91dc88f09650d5c307b00"
EXPORT_FROM_SOURCE = False  # Set to True to export directly

print(f"📋 Dashboard JSON Location: {DASHBOARD_JSON_PATH}")
print(f"📤 Export from source: {EXPORT_FROM_SOURCE}")

In [ ]:
# Load dashboard JSON
import json
from pathlib import Path

if EXPORT_FROM_SOURCE:
    # Export directly from source workspace
    from databricks.sdk import WorkspaceClient
    
    source_client = WorkspaceClient(
        host=SOURCE_WORKSPACE_URL,
        token=SOURCE_PAT_TOKEN if AUTH_METHOD == "pat" else None
    )
    
    print(f"📤 Exporting dashboard {SOURCE_DASHBOARD_ID} from {SOURCE_WORKSPACE_URL}...")
    dashboard = source_client.lakeview.get(dashboard_id=SOURCE_DASHBOARD_ID)
    dashboard_json = json.loads(dashboard.serialized_dashboard)
    print(f"✅ Dashboard exported: {dashboard.display_name}")
else:
    # Load from file
    if DASHBOARD_JSON_PATH.startswith("/Volumes/") or DASHBOARD_JSON_PATH.startswith("/dbfs/"):
        # For Unity Catalog Volumes or DBFS, use dbutils
        with open(DASHBOARD_JSON_PATH, 'r') as f:
            dashboard_json = json.load(f)
    else:
        # For workspace files, use Path
        with open(DASHBOARD_JSON_PATH, 'r') as f:
            dashboard_json = json.load(f)
    
    print(f"✅ Dashboard JSON loaded from: {DASHBOARD_JSON_PATH}")

print(f"\n📊 Dashboard Name: {dashboard_json.get('display_name', 'N/A')}")
print(f"📊 Dashboard Engine: {dashboard_json.get('engine', 'N/A')}")

## 🚀 Step 3: Migrate Dashboard

Choose your migration method:

### Option A: Manual Method
Export JSON from old workspace, import into new workspace, and validate the dashboard.

### Option B: Script Method  
Use the automated migration script (`migrate_dashboard.py`) for a streamlined migration process.

---


### Option A: Manual Method - Export and Import Dashboard

This method allows you to manually control each step of the migration process.

In [ ]:
# ============================================================================
# STEP 3A: MANUAL METHOD - Export Dashboard from Source Workspace
# ============================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard
import json
from datetime import datetime

# Source workspace configuration
SOURCE_WORKSPACE_URL = "https://e2-demo-field-eng.cloud.databricks.com"
SOURCE_DASHBOARD_ID = "01f0fb1aabc91dc88f09650d5c307b00"

# Create source workspace client
source_client = WorkspaceClient(
    host=SOURCE_WORKSPACE_URL,
    token=SOURCE_PAT_TOKEN if AUTH_METHOD == "pat" else None
)

print(f"📤 Exporting dashboard from source workspace: {SOURCE_WORKSPACE_URL}")
print(f"   Dashboard ID: {SOURCE_DASHBOARD_ID}\n")

try:
    # Export dashboard
    dashboard = source_client.lakeview.get(dashboard_id=SOURCE_DASHBOARD_ID)
    
    # Parse dashboard JSON
    dashboard_json = json.loads(dashboard.serialized_dashboard)
    
    print(f"✅ Dashboard exported successfully!")
    print(f"   Name: {dashboard.display_name}")
    print(f"   Warehouse ID: {dashboard.warehouse_id}")
    print(f"   ETag: {dashboard.etag}")
    
    # Save exported JSON to file (optional)
    export_filename = f"dashboard_export_{SOURCE_DASHBOARD_ID}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    export_path = f"/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/{export_filename}"
    
    with open(export_path, 'w') as f:
        json.dump(dashboard_json, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Dashboard JSON saved to: {export_path}")
    
except Exception as e:
    print(f"❌ Error exporting dashboard: {e}")
    raise

### Step 3A.2: Rewrite Catalog.Schema.Table References

Before importing, update all catalog.schema.table references to match the new workspace structure.

In [ ]:
# ============================================================================
# STEP 3A.2: Rewrite Catalog.Schema.Table References
# ============================================================================

import re
from typing import Dict, Tuple

# Define catalog/schema mappings
# Format: (old_catalog, old_schema) -> (new_catalog, new_schema)
CATALOG_SCHEMA_MAP = {
    ("demo_catalog", "demo_schema"): ("archana_krish_fe_dsa", "vizient_deep_dive"),
    # Add more mappings as needed:
    # ("old_catalog", "old_schema"): ("new_catalog", "new_schema"),
}

def rewrite_references(dashboard_json: dict, catalog_schema_map: Dict[Tuple[str, str], Tuple[str, str]]) -> dict:
    """
    Rewrite catalog/schema references in dashboard JSON.
    
    Args:
        dashboard_json: Dashboard JSON structure
        catalog_schema_map: Mapping of (old_catalog, old_schema) -> (new_catalog, new_schema)
    
    Returns:
        Rewritten dashboard JSON
    """
    if not catalog_schema_map:
        print("⚠️  No catalog/schema mappings provided. Skipping rewrite.")
        return dashboard_json
    
    def rewrite_sql(sql_string: str) -> str:
        """Rewrite SQL string with new catalog/schema references."""
        rewritten = sql_string
        for (old_catalog, old_schema), (new_catalog, new_schema) in catalog_schema_map.items():
            # Pattern 1: Backtick-quoted identifiers: `catalog`.`schema`.`table`
            pattern1 = rf"`{re.escape(old_catalog)}`\.`{re.escape(old_schema)}`\."
            replacement1 = f"`{new_catalog}`.`{new_schema}`."
            rewritten = re.sub(pattern1, replacement1, rewritten, flags=re.IGNORECASE)
            
            # Pattern 2: Bare identifiers: catalog.schema.table
            pattern2 = rf"\b{re.escape(old_catalog)}\.{re.escape(old_schema)}\."
            replacement2 = f"{new_catalog}.{new_schema}."
            rewritten = re.sub(pattern2, replacement2, rewritten, flags=re.IGNORECASE)
        return rewritten
    
    def walk_and_rewrite(obj):
        """Recursively walk JSON structure and rewrite SQL references."""
        if isinstance(obj, dict):
            result = {}
            for key, value in obj.items():
                # Check if this is a SQL query field
                if isinstance(value, str) and (
                    key in ("query", "sql", "statement") or "query" in key.lower() or "expression" in key.lower()
                ):
                    result[key] = rewrite_sql(value)
                else:
                    result[key] = walk_and_rewrite(value)
            return result
        elif isinstance(obj, list):
            return [walk_and_rewrite(item) for item in obj]
        else:
            return obj
    
    return walk_and_rewrite(dashboard_json)

# Display mappings
print("🔄 Catalog/Schema Mappings:")
for (old_cat, old_sch), (new_cat, new_sch) in CATALOG_SCHEMA_MAP.items():
    print(f"   {old_cat}.{old_sch} → {new_cat}.{new_sch}")

# Rewrite references
print(f"\n🔄 Rewriting catalog/schema references in dashboard JSON...")
updated_dashboard_json = rewrite_references(dashboard_json, CATALOG_SCHEMA_MAP)

# Compare before/after
original_serialized = json.dumps(dashboard_json, separators=(",", ":"), ensure_ascii=False)
updated_serialized = json.dumps(updated_dashboard_json, separators=(",", ":"), ensure_ascii=False)

if original_serialized != updated_serialized:
    print("✅ References updated successfully!")
    
    # Count changes
    changes = 0
    for (old_cat, old_sch), (new_cat, new_sch) in CATALOG_SCHEMA_MAP.items():
        old_pattern = f"{old_cat}.{old_sch}"
        changes += original_serialized.count(old_pattern)
    
    print(f"   Found {changes} reference(s) to update")
else:
    print("ℹ️  No changes detected (references may already be correct or not found)")

# Update the dashboard object with rewritten JSON
dashboard.serialized_dashboard = updated_serialized
dashboard_json = updated_dashboard_json

print("\n✅ Dashboard JSON updated with new catalog/schema references")

In [ ]:
# ============================================================================
# STEP 3A.2 (continued): Save Updated JSON to File
# ============================================================================

# Choose where to save the updated JSON:
# Option 1: Workspace file
SAVE_TO_WORKSPACE = True
WORKSPACE_FILE_PATH = f"/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/dashboard_updated_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

# Option 2: Unity Catalog Volume
SAVE_TO_VOLUME = False
VOLUME_PATH = "/Volumes/archana_krish_fe_dsa/vizient_deep_dive/data_files/dashboard_updated.json"

# Option 3: DBFS
SAVE_TO_DBFS = False
DBFS_PATH = "/dbfs/mnt/dashboards/dashboard_updated.json"

try:
    updated_json_str = json.dumps(updated_dashboard_json, indent=2, ensure_ascii=False)
    
    if SAVE_TO_WORKSPACE:
        with open(WORKSPACE_FILE_PATH, 'w', encoding='utf-8') as f:
            f.write(updated_json_str)
        print(f"💾 Updated dashboard JSON saved to workspace file:")
        print(f"   {WORKSPACE_FILE_PATH}")
        UPDATED_DASHBOARD_PATH = WORKSPACE_FILE_PATH
    
    if SAVE_TO_VOLUME:
        with open(VOLUME_PATH, 'w', encoding='utf-8') as f:
            f.write(updated_json_str)
        print(f"💾 Updated dashboard JSON saved to Unity Catalog Volume:")
        print(f"   {VOLUME_PATH}")
        UPDATED_DASHBOARD_PATH = VOLUME_PATH
    
    if SAVE_TO_DBFS:
        with open(DBFS_PATH, 'w', encoding='utf-8') as f:
            f.write(updated_json_str)
        print(f"💾 Updated dashboard JSON saved to DBFS:")
        print(f"   {DBFS_PATH}")
        UPDATED_DASHBOARD_PATH = DBFS_PATH
    
    print(f"\n✅ Updated JSON ready for import!")
    print(f"   File size: {len(updated_json_str)} bytes")
    
except Exception as e:
    print(f"❌ Error saving updated JSON: {e}")
    raise

### Step 3A.3: Import Dashboard to Target Workspace

Now import the updated dashboard JSON (with rewritten catalog/schema references) to the target workspace.

In [ ]:
# ============================================================================
# STEP 3A.3: Import Dashboard to Target Workspace
# ============================================================================

# Target workspace configuration
TARGET_WORKSPACE_URL = "https://adb-7405609619727450.10.azuredatabricks.net"
TARGET_DASHBOARD_NAME = "Category Insights - Healthcare Supply Chain"
TARGET_DASHBOARD_PATH = "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration"
TARGET_WAREHOUSE_ID = None  # ⚠️ SET THIS: Get from SQL Warehouses in target workspace

# Create target workspace client
target_client = WorkspaceClient(
    host=TARGET_WORKSPACE_URL,
    token=TARGET_PAT_TOKEN if AUTH_METHOD == "pat" else None
)

if not TARGET_WAREHOUSE_ID:
    print("⚠️  WARNING: TARGET_WAREHOUSE_ID not set!")
    print("   Please set it above. You can find it in:")
    print("   Target Workspace → SQL Warehouses → Select warehouse → Settings → Warehouse ID")
    raise ValueError("TARGET_WAREHOUSE_ID must be set")

print(f"📥 Importing dashboard to target workspace: {TARGET_WORKSPACE_URL}")
print(f"   Dashboard Name: {TARGET_DASHBOARD_NAME}")
print(f"   Dashboard Path: {TARGET_DASHBOARD_PATH}")
print(f"   Warehouse ID: {TARGET_WAREHOUSE_ID}\n")

try:
    # Create dashboard object
    new_dashboard = Dashboard.from_dict({
        "display_name": TARGET_DASHBOARD_NAME,
        "parent_path": TARGET_DASHBOARD_PATH,
        "warehouse_id": TARGET_WAREHOUSE_ID,
        "serialized_dashboard": dashboard.serialized_dashboard
    })
    
    # Import dashboard
    created_dashboard = target_client.lakeview.create(dashboard=new_dashboard)
    
    print(f"✅ Dashboard imported successfully!")
    print(f"   New Dashboard ID: {created_dashboard.dashboard_id}")
    print(f"   Dashboard URL: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{created_dashboard.dashboard_id}")
    
    TARGET_DASHBOARD_ID = created_dashboard.dashboard_id
    
except Exception as e:
    print(f"❌ Error importing dashboard: {e}")
    raise

### Step 3A.4: Validate Imported Dashboard

Validate that the imported dashboard works correctly with the new catalog/schema references.

In [ ]:
# ============================================================================
# STEP 3A.4: Validate Imported Dashboard
# ============================================================================

print(f"🔍 Validating imported dashboard: {TARGET_DASHBOARD_ID}\n")

try:
    # Get the imported dashboard
    imported_dashboard = target_client.lakeview.get(dashboard_id=TARGET_DASHBOARD_ID)
    
    print(f"✅ Dashboard validation successful!")
    print(f"   Name: {imported_dashboard.display_name}")
    print(f"   Path: {imported_dashboard.path}")
    print(f"   Warehouse ID: {imported_dashboard.warehouse_id}")
    print(f"   Status: {'Published' if hasattr(imported_dashboard, 'published') and imported_dashboard.published else 'Draft'}")
    
    # Parse dashboard JSON to check datasets
    imported_json = json.loads(imported_dashboard.serialized_dashboard)
    
    if "datasets" in imported_json:
        print(f"\n📊 Dashboard contains {len(imported_json['datasets'])} dataset(s):")
        for idx, dataset in enumerate(imported_json['datasets'], 1):
            print(f"   {idx}. {dataset.get('name', 'Unnamed')}")
    
    if "pages" in imported_json:
        print(f"\n📄 Dashboard contains {len(imported_json['pages'])} page(s):")
        for page_name, page_data in imported_json['pages'].items():
            widgets_count = len(page_data.get('layout', []))
            print(f"   - {page_name}: {widgets_count} widget(s)")
    
    # Verify that references were updated correctly
    imported_serialized = json.dumps(imported_json, separators=(",", ":"), ensure_ascii=False)
    print(f"\n🔍 Verifying catalog/schema references...")
    
    # Check if old references still exist (they shouldn't)
    old_refs_found = []
    for (old_cat, old_sch), (new_cat, new_sch) in CATALOG_SCHEMA_MAP.items():
        old_pattern = f"{old_cat}.{old_sch}"
        if old_pattern in imported_serialized:
            old_refs_found.append(old_pattern)
    
    if old_refs_found:
        print(f"⚠️  Warning: Found old references that weren't updated:")
        for ref in old_refs_found:
            print(f"   - {ref}")
    else:
        print(f"✅ All catalog/schema references verified (old references not found)")
    
    # Check if new references exist
    new_refs_found = []
    for (old_cat, old_sch), (new_cat, new_sch) in CATALOG_SCHEMA_MAP.items():
        new_pattern = f"{new_cat}.{new_sch}"
        if new_pattern in imported_serialized:
            new_refs_found.append(new_pattern)
    
    if new_refs_found:
        print(f"✅ Found new catalog/schema references:")
        for ref in set(new_refs_found):
            print(f"   - {ref}")
    
    print(f"\n🌐 Open dashboard: {TARGET_WORKSPACE_URL}/sql/dashboardsv3/{TARGET_DASHBOARD_ID}")
    print(f"\n💡 Next steps:")
    print(f"   1. Open the dashboard URL above")
    print(f"   2. Verify all visuals load correctly")
    print(f"   3. Check that data is displaying from the correct tables")
    print(f"   4. Test filters and interactions")
    
except Exception as e:
    print(f"❌ Error validating dashboard: {e}")
    raise

### Option B: Script Method - Automated Migration

Use the `migrate_dashboard.py` script for automated migration with config file support.

#### Script Location and Details

- **Script Name**: `migrate_dashboard.py`
- **Location**: `/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/migrate_dashboard.py`
- **Config File Examples**: 
  - `migration_config.json.example` (general template)
  - `migration_config_dev.json.example` (dev environment)
  - `migration_config_prod.json.example` (production environment)

#### Features of the Script:
- ✅ Automated export and import
- ✅ Catalog/schema reference rewriting
- ✅ Query validation before import
- ✅ Config file support for environment-agnostic deployment
- ✅ PAT and OAuth authentication support
- ✅ Backup creation
- ✅ Dashboard publishing option

In [ ]:
# ============================================================================
# STEP 3B: SCRIPT METHOD - Prepare Configuration File
# ============================================================================

# First, create your configuration file based on the example
# You can copy migration_config.json.example and customize it

import json
from pathlib import Path

# Configuration file path
CONFIG_FILE_PATH = "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/migration_config.json"

# Example configuration (customize as needed)
config_example = {
    "source": {
        "workspace": "https://e2-demo-field-eng.cloud.databricks.com",
        "dashboard_id": "01f0fb1aabc91dc88f09650d5c307b00",
        "pat_token": "${SOURCE_PAT_TOKEN}"  # Use environment variable
    },
    "target": {
        "workspace": "https://adb-7405609619727450.10.azuredatabricks.net",
        "dashboard_name": "Category Insights - Healthcare Supply Chain",
        "path": "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration",
        "warehouse_id": "${TARGET_WAREHOUSE_ID}",  # Use environment variable
        "pat_token": "${TARGET_PAT_TOKEN}"  # Use environment variable
    },
    "auth_method": "pat",
    "validate_queries": True,
    "publish": False,
    "create_backup": True,
    "catalog_schema_map": {
        "demo_catalog.demo_schema": "archana_krish_fe_dsa.vizient_deep_dive"
    }
}

print("📋 Configuration file template:")
print(json.dumps(config_example, indent=2))

print(f"\n💡 To use the script:")
print(f"   1. Create {CONFIG_FILE_PATH} with your settings")
print(f"   2. Set environment variables: SOURCE_PAT_TOKEN, TARGET_PAT_TOKEN, TARGET_WAREHOUSE_ID")
print(f"   3. Run: python {CONFIG_FILE_PATH.replace('migration_config.json', 'migrate_dashboard.py')} --config-file {CONFIG_FILE_PATH}")

In [ ]:
# ============================================================================
# STEP 3B (continued): Run Migration Script
# ============================================================================

# Option 1: Run script from notebook using %sh magic command
# Uncomment and customize the command below:

SCRIPT_PATH = "/Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/migrate_dashboard.py"

print("🚀 To run the migration script, use one of these methods:\n")

print("Method 1: Run from terminal/command line:")
print(f"   export SOURCE_PAT_TOKEN='your-source-token'")
print(f"   export TARGET_PAT_TOKEN='your-target-token'")
print(f"   export TARGET_WAREHOUSE_ID='your-warehouse-id'")
print(f"   python {SCRIPT_PATH} --config-file {CONFIG_FILE_PATH}\n")

print("Method 2: Run from this notebook (uncomment below):")
print(f"   %sh python {SCRIPT_PATH} --config-file {CONFIG_FILE_PATH}\n")

print("Method 3: With CLI overrides:")
print(f"   python {SCRIPT_PATH} --config-file {CONFIG_FILE_PATH} --publish --validate-queries\n")

# Uncomment to run from notebook:
# %sh 
# export SOURCE_PAT_TOKEN=$SOURCE_PAT_TOKEN
# export TARGET_PAT_TOKEN=$TARGET_PAT_TOKEN  
# export TARGET_WAREHOUSE_ID=$TARGET_WAREHOUSE_ID
# python /Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/migrate_dashboard.py --config-file /Workspace/Users/archana.krishnamurthy@databricks.com/01-Customer-Projects/Vizient/Dashboard-Migration/migration_config.json

#### Script Method - Additional Options

The script supports many command-line options that override config file settings:

```bash
# Basic usage with config file
python migrate_dashboard.py --config-file migration_config.json

# Override specific settings
python migrate_dashboard.py \
  --config-file migration_config.json \
  --target-dashboard-name "Custom Name" \
  --publish \
  --validate-queries

# Use OAuth instead of PAT
python migrate_dashboard.py \
  --config-file migration_config.json \
  --auth-method oauth

# Full CLI without config file (backward compatible)
python migrate_dashboard.py \
  --source-workspace https://workspace1.cloud.databricks.com \
  --source-dashboard-id 01abc123 \
  --source-pat-token dapi123... \
  --target-workspace https://workspace2.cloud.databricks.com \
  --target-dashboard-name "My Dashboard" \
  --target-path "/Workspace/Users/user@example.com/Dashboards" \
  --target-warehouse-id abc123 \
  --target-pat-token dapi456... \
  --validate-queries \
  --publish
```

For complete documentation, see: `DASHBOARD_MIGRATION_README.md`

## 🔍 Step 2: Analyze JSON and Extract Catalog.Schema.Table References

This step will:
1. Extract all `catalog.schema.table` references from the dashboard JSON
2. Check if these objects exist in the target workspace
3. Report any missing objects or permission issues

In [ ]:
# ============================================================================
# STEP 2: EXTRACT CATALOG.SCHEMA.TABLE REFERENCES
# ============================================================================

import re
from typing import List, Tuple, Set

def extract_table_references(dashboard_json: dict) -> List[Tuple[str, str, str]]:
    """
    Extract all catalog.schema.table references from dashboard JSON.
    
    Returns:
        List of (catalog, schema, table) tuples
    """
    references = set()
    
    def extract_from_value(value, path=""):
        if isinstance(value, str):
            # Pattern 1: Backtick-quoted identifiers: `catalog`.`schema`.`table`
            pattern1 = r'`([a-zA-Z0-9_]+)`\.`([a-zA-Z0-9_]+)`\.`([a-zA-Z0-9_]+)`'
            matches1 = re.findall(pattern1, value)
            for match in matches1:
                if len(match) == 3:
                    references.add((match[0], match[1], match[2]))
            
            # Pattern 2: Bare identifiers: catalog.schema.table
            pattern2 = r'\b([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)\b'
            matches2 = re.findall(pattern2, value)
            for match in matches2:
                if len(match) == 3:
                    references.add((match[0], match[1], match[2]))
        elif isinstance(value, dict):
            for key, val in value.items():
                extract_from_value(val, f"{path}.{key}" if path else key)
        elif isinstance(value, list):
            for idx, item in enumerate(value):
                extract_from_value(item, f"{path}[{idx}]" if path else f"[{idx}]")
    
    extract_from_value(dashboard_json)
    return sorted(references)

# Extract all references
table_references = extract_table_references(dashboard_json)

print(f"\n🔍 Found {len(table_references)} catalog.schema.table references:\n")
for catalog, schema, table in table_references:
    print(f"   📊 {catalog}.{schema}.{table}")

if len(table_references) == 0:
    print("\n⚠️  No catalog.schema.table references found in dashboard JSON.")
    print("   The dashboard might use views, temporary tables, or different reference patterns.")

In [ ]:
# ============================================================================
# STEP 2 (continued): CHECK IF OBJECTS EXIST IN TARGET WORKSPACE
# ============================================================================

from databricks.sdk import WorkspaceClient

# Target workspace configuration
TARGET_WORKSPACE_URL = "https://adb-7405609619727450.10.azuredatabricks.net"

# Create target workspace client
target_client = WorkspaceClient(
    host=TARGET_WORKSPACE_URL,
    token=TARGET_PAT_TOKEN if AUTH_METHOD == "pat" else None
)

print(f"\n🔍 Checking objects in target workspace: {TARGET_WORKSPACE_URL}\n")

# Check each table reference
validation_results = []

for catalog, schema, table in table_references:
    full_name = f"{catalog}.{schema}.{table}"
    
    try:
        # Try to get table info
        table_info = target_client.tables.get(full_name=full_name)
        
        validation_results.append({
            "catalog": catalog,
            "schema": schema,
            "table": table,
            "exists": True,
            "table_type": table_info.table_type,
            "data_source_format": getattr(table_info, 'data_source_format', 'N/A'),
            "status": "✅ EXISTS"
        })
        
        print(f"✅ {full_name} - EXISTS ({table_info.table_type})")
        
    except Exception as e:
        # Check if it's a permission issue or object doesn't exist
        error_msg = str(e)
        
        if "does not exist" in error_msg.lower() or "not found" in error_msg.lower():
            status = "❌ NOT FOUND"
        elif "permission" in error_msg.lower() or "access" in error_msg.lower():
            status = "⚠️  PERMISSION DENIED"
        else:
            status = f"❓ ERROR: {error_msg[:50]}"
        
        validation_results.append({
            "catalog": catalog,
            "schema": schema,
            "table": table,
            "exists": False,
            "error": error_msg,
            "status": status
        })
        
        print(f"{status} - {full_name}")

print(f"\n{'='*80}")
print(f"📊 VALIDATION SUMMARY")
print(f"{'='*80}")

existing_count = sum(1 for r in validation_results if r["exists"])
missing_count = len(validation_results) - existing_count

print(f"\n✅ Objects found: {existing_count}/{len(validation_results)}")
print(f"❌ Objects missing: {missing_count}/{len(validation_results)}")

if missing_count > 0:
    print(f"\n⚠️  ACTION REQUIRED:")
    print(f"   The following objects need to be migrated or mapped:")
    for result in validation_results:
        if not result["exists"]:
            print(f"   - {result['catalog']}.{result['schema']}.{result['table']}")

In [ ]:
# Display detailed results as a table
import pandas as pd

df_results = pd.DataFrame(validation_results)
display(df_results)

## 📝 Next Steps

Based on the analysis above:

1. **If all objects exist**: Proceed with dashboard migration
2. **If some objects are missing**:
   - Migrate the missing tables/views to the target workspace
   - Or create a catalog/schema mapping in the migration script
   - Update the dashboard JSON to reference the new locations

3. **If permission issues**:
   - Grant appropriate permissions in Unity Catalog
   - Ensure the service principal/user has access to the required catalogs/schemas

---

**Continue to the next steps in this notebook as you add them...**